# 模块 G · 空间趋势：以某个区域为中心，基因 / 细胞组成随距离如何变化   [Python，需要 SOAPy_st]

- 对应课程：第10课（课程叫"细胞轨迹"，实际是空间距离上的趋势，不是发育轨迹）
- 适用：所有 RNA 平台；最常见的用法是"肿瘤边界向外，免疫细胞如何变化"
- 注意：SOAPy 的距离单位是 obsm['spatial'] 的单位，这里换成 µm；radius / cut 都按 µm 理解。

**对应的流程脚本**：`scripts/G_special/spatial_tendency_soapy.py`（代码相同，流程运行的是 .py）

**用法**：先把下面第一个代码格里的 `project_dir` 改成项目目录，然后从上到下逐格运行。

In [ ]:
# 步骤：切换到项目目录（里面有 config.yaml、samples.tsv 和 results/）
# 输入：project_dir（改成自己的项目目录）
# 输出：无（之后的代码都以项目目录为当前目录）
import os
project_dir = "."      # 例如 "/data/my_visium_project"
os.chdir(project_dir)
os.getcwd()

## 1. 导入包

In [ ]:
# 步骤：导入包
# 输入：无
# 输出：无
from pathlib import Path
import matplotlib.pyplot as plt
import scanpy as sc
import SOAPy_st as sp
import yaml

## 2. 读取配置，指定中心区域

In [ ]:
# 步骤：读取配置，指定中心区域
# 输入：config.yaml
# 输出：cfg、out_dir、center_key、center_value、radius_um
cfg = yaml.safe_load(open("config.yaml"))
out_dir = Path(cfg["out_dir"])
center_key = "niche"          # 用哪个分组定义中心区域（niche / domain / cell_type）
center_value = "niche_1"      # 中心区域，比如肿瘤核心所在的 niche
radius_um = 1000              # 从区域边界向外看多远

这一步得到：cfg、out_dir、center_key、center_value、radius_um

## 3. 读入数据，取一个样本（区域掩膜是按单张切片画的），坐标换成 µm

In [ ]:
# 步骤：读入数据，取一个样本（区域掩膜是按单张切片画的），坐标换成 µm
# 输入：results/05_niche.h5ad
# 输出：adata（单样本）
adata = sc.read_h5ad(out_dir / "05_niche.h5ad")
sample = adata.obs["sample"].unique()[0]
adata = adata[adata.obs["sample"] == sample].copy()
adata.obsm["spatial"] = adata.obsm["spatial_um"]

这一步得到：adata（单样本）

## 4. 根据中心区域生成掩膜

In [ ]:
# 步骤：根据中心区域生成掩膜
# 输入：adata
# 输出：mask（二维数组）；图 results/tendency_mask.png
mask = sp.tl.get_mask_from_domain(adata, clusters=center_value, KSize=35, cluster_key=center_key)
plt.imshow(mask, cmap="gray")
plt.savefig(out_dir / "tendency_mask.png", bbox_inches="tight")

这一步得到：mask（二维数组）；图 results/tendency_mask.png

## 5. 哪些基因随离区域的距离显著变化（Spearman 相关）

In [ ]:
# 步骤：哪些基因随离区域的距离显著变化（Spearman 相关）
# 输入：adata、mask
# 输出：表 results/tendency_spearman.csv
spearman = sp.tl.spearman_correlation(adata, mask, radius=radius_um, num=5)
spearman.to_csv(out_dir / "tendency_spearman.csv")
spearman.head(10)

这一步得到：表 results/tendency_spearman.csv

## 6. 拟合每个基因随距离的变化曲线，并把曲线相似的基因聚成一类

In [ ]:
# 步骤：拟合每个基因随距离的变化曲线，并把曲线相似的基因聚成一类
# 输入：adata、mask
# 输出：图 results/tendency_gene_clusters.png
sp.tl.spatial_tendency(adata, mask, radius=radius_um, location="out", frac=5)
sp.tl.gene_cluster(adata=adata, k=8, range_min=0.03, fdr=True, pvalue=0.05)
sp.pl.show_curves_cluster(adata)
plt.savefig(out_dir / "tendency_gene_clusters.png", bbox_inches="tight")

这一步得到：图 results/tendency_gene_clusters.png